# 3.00 - Summary Statistics for plot

- Counting airports with the most haunted places intersections and saving to json in ./data/processed. 
- Data used in 3.02 visualization

**../data/processed/flight_haunted_place_counts.json**

- Key flight name {[sourceIATA] -> [destIATA]}
- Value is \# of haunted places within radius

**../data/processed/airport_haunted_place_counts.json**

- Key is full name of airport
- Value is \# of haunted places within radius


In [1]:
# System Path #
import os
import sys 

parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

# Pandas, json and runtime #
import pandas as pd
from tqdm import tqdm
import json
import time
tqdm.pandas()

# Tika #
import tika as tk
from tika import parser

# Iterators #
from collections import Counter
from itertools import chain
import re

# local functions #
from dsci_550_a1.parsingFunctions import extractSequences
from dsci_550_a1.unpack_circles import unpack_cluster


## Features Added df_haunted_places ##
outfile = "../data/processed/haunted_places_features_added.tab"

df_haunted_places = pd.read_csv(outfile, sep = "\t")

## Flight Intersection Data
with open("../data/processed/flight_proximity_data.json") as f:
    flight_intersection_data = json.load(f)
    
## Airport Intersection Data
with open("../data/processed/airport_proximity_data.json") as f:
    airport_intersection_data = json.load(f)

In [2]:

df_american_airports = pd.read_csv("../data/joined_datasets/american_airports.tsv", sep = "\t")
airport_counter = Counter()

for i, data in airport_intersection_data.items():
    for airport in data["Airports"]:
        airport_counter[airport["Name"]] += 1

airport_counts_by_type = {}
for type in pd.unique(df_american_airports['Type']).tolist():

    airport_names_with_type = df_american_airports.loc[df_american_airports['Type'] == f'{type}', 'Name'].tolist()
    filtered_counter = Counter()

    for key, val in airport_counter.most_common():
        if key in airport_names_with_type:
            filtered_counter[key] = val 


    airport_counts_by_type[type] = filtered_counter.most_common()

with open('../data/processed/airport_haunted_place_counts.json', 'w') as f:
    json.dump(dict(airport_counts_by_type), f, indent=4)


## Counting most common Flights ##
flight_counter = Counter()
for i, data in flight_intersection_data.items():
    
    for flight in data["Routes"]:
        source_airport, dest_airport = flight["Source_Airport"], flight["Dest_Airport"]
        flight_name = f"{source_airport} -> {dest_airport}"
        flight_counter[flight_name] += 1

with open('../data/processed/flight_haunted_place_counts.json', 'w') as f:
    json.dump(dict(flight_counter.most_common()), f, indent=4)